# 10 — Três Linhas de Pesquisa para Segmentação Municipal

**Objetivo:** Aprofundar a clusterização dos 5.204 municípios brasileiros através de 3 abordagens complementares, partindo do modelo base K=4.

| Linha | Abordagem | Descrição |
|-------|-----------|----------|
| 1 | Hierárquica | Subclusters dentro de cada grupo K=4 |
| 2 | K=10 | Mais grupos diretamente |
| 3 | Regional | KMeans+PCA por região (K automático) |

**Tratamento de outliers:** removidos para fit, reassociados ao cluster mais próximo após treinamento.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

BASE = r'c:\Users\guilhermecorrea\Downloads\Gui\Projetos\PortifolioProjetos\projeto_5'
OUT_RES = os.path.join(BASE, 'outputs', 'results')
OUT_FIG = os.path.join(BASE, 'outputs', 'figures')

print('Setup concluído.')

Setup concluído.


## 0. Dados e Modelo Base K=4

In [2]:
# Carregar dados
df = pd.read_csv(os.path.join(OUT_RES, 'municipios_clusters_melhorado.csv'))
print(f'Total municípios: {len(df)}')

features_v3 = [
    'log_populacao', 'log_densidade', 'pib_per_capita',
    'taxa_alfabetizacao', 'mortalidade_infantil', 'esgoto_adequado',
    'saude_per_10k', 'indice_desenvolvimento_social',
    'indice_infraestrutura', 'urbanizacao_proxy'
]

X_raw = df[features_v3].values

# Detectar outliers via percentis (P1/P99) — mesmo método do notebook 09
outlier_mask = np.zeros(len(df), dtype=bool)
for col in ['populacao_original', 'pib_per_capita_original', 'densidade_original']:
    p1, p99 = df[col].quantile(0.01), df[col].quantile(0.99)
    outlier_mask |= (df[col] < p1) | (df[col] > p99)

n_outliers = outlier_mask.sum()
print(f'Outliers detectados: {n_outliers} ({n_outliers/len(df)*100:.1f}%)')
print(f'Municípios para fit: {(~outlier_mask).sum()}')

df['is_outlier'] = outlier_mask

Total municípios: 5204
Outliers detectados: 298 (5.7%)
Municípios para fit: 4906


In [3]:
# Fit K=4 sem outliers, depois reassociar outliers
scaler = StandardScaler()
X_fit = scaler.fit_transform(df.loc[~outlier_mask, features_v3])
X_all = scaler.transform(X_raw)

km4 = KMeans(n_clusters=4, random_state=42, n_init=20, max_iter=300)
km4.fit(X_fit)

# Atribuir todos os municípios (incluindo outliers)
df['cluster_k4'] = km4.predict(X_all)

sil = silhouette_score(X_all, df['cluster_k4'])
db = davies_bouldin_score(X_all, df['cluster_k4'])
cal = calinski_harabasz_score(X_all, df['cluster_k4'])

print(f'\nK=4 — Métricas (todos os {len(df)} municípios):')
print(f'  Silhueta: {sil:.4f}')
print(f'  Davies-Bouldin: {db:.4f}')
print(f'  Calinski-Harabasz: {cal:.1f}')
print(f'\nDistribuição:')
for c in range(4):
    n = (df['cluster_k4'] == c).sum()
    n_out = ((df['cluster_k4'] == c) & df['is_outlier']).sum()
    print(f'  Grupo {c}: {n} municípios ({n/len(df)*100:.1f}%) — {n_out} outliers reassociados')


K=4 — Métricas (todos os 5204 municípios):
  Silhueta: 0.1585
  Davies-Bouldin: 1.6845
  Calinski-Harabasz: 1009.4

Distribuição:
  Grupo 0: 597 municípios (11.5%) — 102 outliers reassociados
  Grupo 1: 1894 municípios (36.4%) — 62 outliers reassociados
  Grupo 2: 807 municípios (15.5%) — 77 outliers reassociados
  Grupo 3: 1906 municípios (36.6%) — 57 outliers reassociados


---
## Linha 1 — Clusterização Hierárquica dentro de cada Grupo K=4

Aplica Agglomerative Clustering com K=4 dentro de cada um dos 4 grupos, gerando 16 subclusters.

In [4]:
df['subcluster_L1'] = -1
metricas_L1 = {}

for grupo in range(4):
    mask = df['cluster_k4'] == grupo
    idx = df.index[mask]
    X_grupo = X_all[mask]
    n_grupo = len(idx)
    
    # K=4 subclusters (ou menos se grupo for pequeno)
    k_sub = min(4, n_grupo // 20)  # mínimo 20 por subcluster
    k_sub = max(2, k_sub)
    
    agg = AgglomerativeClustering(n_clusters=k_sub, linkage='ward')
    labels = agg.fit_predict(X_grupo)
    
    # Subcluster global: grupo*4 + sub
    df.loc[idx, 'subcluster_L1'] = grupo * 4 + labels
    
    if k_sub >= 2:
        sil_g = silhouette_score(X_grupo, labels)
    else:
        sil_g = 0
    
    metricas_L1[grupo] = {
        'n': n_grupo,
        'k_sub': k_sub,
        'silhouette': round(sil_g, 4),
        'distribuicao': {int(k): int(v) for k, v in pd.Series(labels).value_counts().sort_index().items()}
    }
    
    print(f'Grupo {grupo} ({n_grupo} munic.) → {k_sub} subclusters, Silhueta interna: {sil_g:.4f}')
    for s in range(k_sub):
        ns = (labels == s).sum()
        print(f'  Sub {s}: {ns} ({ns/n_grupo*100:.1f}%)')

n_subclusters_L1 = df['subcluster_L1'].nunique()
sil_L1_global = silhouette_score(X_all, df['subcluster_L1'])
print(f'\nLinha 1 — Total: {n_subclusters_L1} subclusters, Silhueta global: {sil_L1_global:.4f}')

Grupo 0 (597 munic.) → 4 subclusters, Silhueta interna: 0.1987
  Sub 0: 358 (60.0%)
  Sub 1: 66 (11.1%)
  Sub 2: 155 (26.0%)
  Sub 3: 18 (3.0%)
Grupo 1 (1894 munic.) → 4 subclusters, Silhueta interna: 0.0532
  Sub 0: 769 (40.6%)
  Sub 1: 215 (11.4%)
  Sub 2: 631 (33.3%)
  Sub 3: 279 (14.7%)
Grupo 2 (807 munic.) → 4 subclusters, Silhueta interna: 0.1110
  Sub 0: 273 (33.8%)
  Sub 1: 325 (40.3%)
  Sub 2: 110 (13.6%)
  Sub 3: 99 (12.3%)


Grupo 3 (1906 munic.) → 4 subclusters, Silhueta interna: 0.0673
  Sub 0: 846 (44.4%)
  Sub 1: 427 (22.4%)
  Sub 2: 375 (19.7%)
  Sub 3: 258 (13.5%)



Linha 1 — Total: 16 subclusters, Silhueta global: 0.0512


In [5]:
# Dendrogramas por grupo
fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor='#111827')
for grupo, ax in enumerate(axes.flat):
    ax.set_facecolor('#111827')
    mask = df['cluster_k4'] == grupo
    X_g = X_all[mask]
    # Amostra para dendrograma legível
    if len(X_g) > 300:
        rng = np.random.RandomState(42)
        sample_idx = rng.choice(len(X_g), 300, replace=False)
        X_sample = X_g[sample_idx]
        titulo = f'Grupo {grupo} (amostra 300/{len(X_g)})'
    else:
        X_sample = X_g
        titulo = f'Grupo {grupo} ({len(X_g)} munic.)'
    
    Z = linkage(X_sample, method='ward')
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=20, 
               leaf_rotation=90, leaf_font_size=8,
               above_threshold_color='#F28C28',
               color_threshold=0)
    ax.set_title(titulo, color='#E5E7EB', fontsize=12, fontweight='bold')
    ax.tick_params(colors='#94A3B8')
    for spine in ax.spines.values():
        spine.set_color('#374151')

fig.suptitle('Dendrogramas — Linha 1 (Hierárquica por Grupo)', 
             color='#E5E7EB', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT_FIG, 'dendrogramas_L1.png'), dpi=150, 
            bbox_inches='tight', facecolor='#111827')
plt.close(fig)
print('Dendrogramas salvos.')

Dendrogramas salvos.


In [6]:
# Perfis detalhados Linha 1
perfis_L1 = {}
for sc in sorted(df['subcluster_L1'].unique()):
    sub = df[df['subcluster_L1'] == sc]
    grupo_pai = sc // 4
    sub_id = sc % 4
    perfis_L1[int(sc)] = {
        'grupo_pai': grupo_pai,
        'sub_id': sub_id,
        'n': len(sub),
        'pct_total': round(len(sub)/len(df)*100, 1),
        'pib_pc_mediano': round(sub['pib_per_capita_original'].median(), 0),
        'pop_mediana': round(sub['populacao_original'].median(), 0),
        'ids_medio': round(sub['indice_desenvolvimento_social'].mean(), 3),
        'esgoto_medio': round(sub['esgoto_adequado_original'].mean(), 1),
        'mortalidade_media': round(sub['mortalidade_infantil_original'].mean(), 1),
        'alfabetizacao_media': round(sub['taxa_alfabetizacao_original'].mean(), 1),
        'top_regioes': sub['regiao'].value_counts().head(3).to_dict(),
        'top_ufs': sub['uf'].value_counts().head(5).to_dict(),
        'n_outliers': int(sub['is_outlier'].sum())
    }

print(f'Linha 1: {len(perfis_L1)} subclusters')
for sc, p in perfis_L1.items():
    print(f"  G{p['grupo_pai']}-S{p['sub_id']}: {p['n']} munic., PIB R${p['pib_pc_mediano']:,.0f}, IDS={p['ids_medio']:.3f}, Pop med={p['pop_mediana']:,.0f}")

Linha 1: 16 subclusters
  G0-S0: 358 munic., PIB R$30,682, IDS=0.454, Pop med=50,891
  G0-S1: 66 munic., PIB R$38,317, IDS=0.452, Pop med=240,900
  G0-S2: 155 munic., PIB R$36,111, IDS=0.464, Pop med=108,622
  G0-S3: 18 munic., PIB R$139,517, IDS=0.545, Pop med=55,999
  G1-S0: 769 munic., PIB R$22,733, IDS=0.510, Pop med=14,297
  G1-S1: 215 munic., PIB R$76,077, IDS=0.533, Pop med=11,355
  G1-S2: 631 munic., PIB R$18,929, IDS=0.532, Pop med=10,601
  G1-S3: 279 munic., PIB R$21,041, IDS=0.478, Pop med=17,520
  G2-S0: 273 munic., PIB R$30,581, IDS=0.511, Pop med=3,398
  G2-S1: 325 munic., PIB R$22,035, IDS=0.440, Pop med=3,815
  G2-S2: 110 munic., PIB R$87,545, IDS=0.493, Pop med=4,135
  G2-S3: 99 munic., PIB R$39,125, IDS=0.456, Pop med=2,355
  G3-S0: 846 munic., PIB R$15,953, IDS=0.413, Pop med=14,170
  G3-S1: 427 munic., PIB R$17,028, IDS=0.353, Pop med=7,450
  G3-S2: 375 munic., PIB R$29,240, IDS=0.389, Pop med=6,892
  G3-S3: 258 munic., PIB R$14,011, IDS=0.371, Pop med=17,210


---
## Linha 2 — KMeans K=10

In [7]:
# Fit K=10 sem outliers, depois reassociar
km10 = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=300)
km10.fit(X_fit)
df['cluster_k10'] = km10.predict(X_all)

sil_k10 = silhouette_score(X_all, df['cluster_k10'])
db_k10 = davies_bouldin_score(X_all, df['cluster_k10'])
cal_k10 = calinski_harabasz_score(X_all, df['cluster_k10'])

print(f'K=10 — Métricas (todos os {len(df)} municípios):')
print(f'  Silhueta: {sil_k10:.4f}')
print(f'  Davies-Bouldin: {db_k10:.4f}')
print(f'  Calinski-Harabasz: {cal_k10:.1f}')
print(f'\nDistribuição:')
for c in range(10):
    n = (df['cluster_k10'] == c).sum()
    n_out = ((df['cluster_k10'] == c) & df['is_outlier']).sum()
    print(f'  Grupo {c}: {n} ({n/len(df)*100:.1f}%) — {n_out} outliers')

K=10 — Métricas (todos os 5204 municípios):
  Silhueta: 0.1215
  Davies-Bouldin: 1.5870
  Calinski-Harabasz: 682.7

Distribuição:
  Grupo 0: 599 (11.5%) — 12 outliers
  Grupo 1: 640 (12.3%) — 15 outliers
  Grupo 2: 483 (9.3%) — 28 outliers
  Grupo 3: 639 (12.3%) — 6 outliers
  Grupo 4: 724 (13.9%) — 27 outliers
  Grupo 5: 481 (9.2%) — 5 outliers
  Grupo 6: 275 (5.3%) — 31 outliers
  Grupo 7: 377 (7.2%) — 64 outliers
  Grupo 8: 706 (13.6%) — 20 outliers
  Grupo 9: 280 (5.4%) — 90 outliers


In [8]:
# Perfis K=10
perfis_L2 = {}
for c in range(10):
    sub = df[df['cluster_k10'] == c]
    perfis_L2[c] = {
        'n': len(sub),
        'pct': round(len(sub)/len(df)*100, 1),
        'pib_pc_mediano': round(sub['pib_per_capita_original'].median(), 0),
        'pop_mediana': round(sub['populacao_original'].median(), 0),
        'ids_medio': round(sub['indice_desenvolvimento_social'].mean(), 3),
        'esgoto_medio': round(sub['esgoto_adequado_original'].mean(), 1),
        'mortalidade_media': round(sub['mortalidade_infantil_original'].mean(), 1),
        'alfabetizacao_media': round(sub['taxa_alfabetizacao_original'].mean(), 1),
        'top_regioes': sub['regiao'].value_counts().head(3).to_dict(),
        'top_ufs': sub['uf'].value_counts().head(5).to_dict(),
        'n_outliers': int(sub['is_outlier'].sum())
    }

print('Perfis K=10:')
for c, p in perfis_L2.items():
    print(f"  Grupo {c}: {p['n']} munic. ({p['pct']}%), PIB R${p['pib_pc_mediano']:,.0f}, IDS={p['ids_medio']:.3f}")

Perfis K=10:
  Grupo 0: 599 munic. (11.5%), PIB R$18,287, IDS=0.463
  Grupo 1: 640 munic. (12.3%), PIB R$19,204, IDS=0.444
  Grupo 2: 483 munic. (9.3%), PIB R$22,559, IDS=0.424
  Grupo 3: 639 munic. (12.3%), PIB R$23,751, IDS=0.564
  Grupo 4: 724 munic. (13.9%), PIB R$16,266, IDS=0.344
  Grupo 5: 481 munic. (9.2%), PIB R$27,888, IDS=0.454
  Grupo 6: 275 munic. (5.3%), PIB R$32,511, IDS=0.475
  Grupo 7: 377 munic. (7.2%), PIB R$81,069, IDS=0.510
  Grupo 8: 706 munic. (13.6%), PIB R$17,470, IDS=0.463
  Grupo 9: 280 munic. (5.4%), PIB R$37,590, IDS=0.462


---
## Linha 3 — Por Região + PCA + KMeans (K automático)

In [9]:
regioes = ['Norte', 'Nordeste', 'Sudeste', 'Sul', 'Centro-Oeste']
resultados_L3 = {}
df['cluster_regional'] = -1
df['regiao_cluster_label'] = ''
offset_label = 0

for regiao in regioes:
    mask_reg = df['regiao'] == regiao
    idx_reg = df.index[mask_reg]
    X_reg_raw = df.loc[mask_reg, features_v3].values
    n_reg = len(idx_reg)
    
    # Outliers dentro da região
    mask_out_reg = df.loc[mask_reg, 'is_outlier'].values
    X_reg_fit = StandardScaler().fit_transform(X_reg_raw[~mask_out_reg])
    
    # PCA — manter ~90% variância
    pca = PCA(n_components=0.90, random_state=42)
    X_pca_fit = pca.fit_transform(X_reg_fit)
    n_comp = pca.n_components_
    var_explained = pca.explained_variance_ratio_.sum()
    
    # Transformar todos (incluindo outliers)
    scaler_reg = StandardScaler()
    scaler_reg.fit(X_reg_raw[~mask_out_reg])
    X_reg_all_scaled = scaler_reg.transform(X_reg_raw)
    X_pca_all = pca.transform(X_reg_all_scaled)
    
    # Determinar K ótimo via Silhouette (testar K=2 a 8)
    best_k, best_sil = 2, -1
    sil_scores = {}
    max_k = min(8, n_reg // 30)
    max_k = max(2, max_k)
    
    for k in range(2, max_k + 1):
        km_test = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels_test = km_test.fit_predict(X_pca_fit)
        s = silhouette_score(X_pca_fit, labels_test)
        sil_scores[k] = round(s, 4)
        if s > best_sil:
            best_sil = s
            best_k = k
    
    # Fit com melhor K e predict todos
    km_reg = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    km_reg.fit(X_pca_fit)
    labels_all = km_reg.predict(X_pca_all)
    
    df.loc[idx_reg, 'cluster_regional'] = offset_label + labels_all
    df.loc[idx_reg, 'regiao_cluster_label'] = [f'{regiao[:2]}-{l}' for l in labels_all]
    
    sil_reg = silhouette_score(X_pca_all, labels_all)
    
    resultados_L3[regiao] = {
        'n_municipios': n_reg,
        'n_outliers': int(mask_out_reg.sum()),
        'n_componentes_pca': n_comp,
        'variancia_explicada': round(var_explained, 4),
        'k_otimo': best_k,
        'silhouette': round(sil_reg, 4),
        'sil_por_k': sil_scores,
        'distribuicao': {int(k): int(v) for k, v in pd.Series(labels_all).value_counts().sort_index().items()}
    }
    
    offset_label += best_k
    
    print(f'\n{regiao} ({n_reg} munic., {int(mask_out_reg.sum())} outliers):')
    print(f'  PCA: {n_comp} componentes ({var_explained:.1%} variância)')
    print(f'  K ótimo: {best_k} (Silhueta: {sil_reg:.4f})')
    print(f'  Silhuetas testadas: {sil_scores}')
    for s in range(best_k):
        ns = (labels_all == s).sum()
        print(f'    Cluster {s}: {ns} ({ns/n_reg*100:.1f}%)')

total_clusters_L3 = offset_label
print(f'\nLinha 3 — Total: {total_clusters_L3} clusters regionais')


Norte (391 munic., 41 outliers):
  PCA: 6 componentes (93.6% variância)
  K ótimo: 2 (Silhueta: 0.2422)
  Silhuetas testadas: {2: 0.2384, 3: 0.1743, 4: 0.1729, 5: 0.1674, 6: 0.1594, 7: 0.1616, 8: 0.1611}
    Cluster 0: 276 (70.6%)
    Cluster 1: 115 (29.4%)



Nordeste (1684 munic., 76 outliers):
  PCA: 6 componentes (90.1% variância)
  K ótimo: 5 (Silhueta: 0.1964)
  Silhuetas testadas: {2: 0.1745, 3: 0.1843, 4: 0.1915, 5: 0.198, 6: 0.1774, 7: 0.1585, 8: 0.1544}
    Cluster 0: 201 (11.9%)
    Cluster 1: 661 (39.3%)
    Cluster 2: 207 (12.3%)
    Cluster 3: 572 (34.0%)
    Cluster 4: 43 (2.6%)



Sudeste (1564 munic., 70 outliers):
  PCA: 6 componentes (92.9% variância)
  K ótimo: 2 (Silhueta: 0.2934)
  Silhuetas testadas: {2: 0.2531, 3: 0.1875, 4: 0.1908, 5: 0.1903, 6: 0.1678, 7: 0.162, 8: 0.1621}
    Cluster 0: 1222 (78.1%)
    Cluster 1: 342 (21.9%)



Sul (1141 munic., 72 outliers):
  PCA: 6 componentes (91.8% variância)
  K ótimo: 3 (Silhueta: 0.1843)
  Silhuetas testadas: {2: 0.1683, 3: 0.1925, 4: 0.1892, 5: 0.176, 6: 0.1631, 7: 0.1581, 8: 0.1566}
    Cluster 0: 494 (43.3%)
    Cluster 1: 470 (41.2%)
    Cluster 2: 177 (15.5%)



Centro-Oeste (424 munic., 39 outliers):
  PCA: 6 componentes (91.7% variância)
  K ótimo: 2 (Silhueta: 0.1751)
  Silhuetas testadas: {2: 0.1897, 3: 0.1783, 4: 0.1661, 5: 0.1585, 6: 0.1555, 7: 0.1604, 8: 0.1517}
    Cluster 0: 140 (33.0%)
    Cluster 1: 284 (67.0%)

Linha 3 — Total: 14 clusters regionais


In [10]:
# Perfis regionais
perfis_L3 = {}
for regiao in regioes:
    sub_reg = df[df['regiao'] == regiao]
    clusters_reg = sorted(sub_reg['cluster_regional'].unique())
    for cr in clusters_reg:
        sub = sub_reg[sub_reg['cluster_regional'] == cr]
        label = sub['regiao_cluster_label'].iloc[0]
        perfis_L3[label] = {
            'regiao': regiao,
            'cluster_id': int(cr),
            'n': len(sub),
            'pct_regiao': round(len(sub)/len(sub_reg)*100, 1),
            'pib_pc_mediano': round(sub['pib_per_capita_original'].median(), 0),
            'pop_mediana': round(sub['populacao_original'].median(), 0),
            'ids_medio': round(sub['indice_desenvolvimento_social'].mean(), 3),
            'esgoto_medio': round(sub['esgoto_adequado_original'].mean(), 1),
            'mortalidade_media': round(sub['mortalidade_infantil_original'].mean(), 1),
            'alfabetizacao_media': round(sub['taxa_alfabetizacao_original'].mean(), 1),
            'top_ufs': sub['uf'].value_counts().head(3).to_dict(),
            'n_outliers': int(sub['is_outlier'].sum())
        }

print(f'Perfis regionais: {len(perfis_L3)} clusters')
for label, p in perfis_L3.items():
    print(f"  {label}: {p['n']} munic., PIB R${p['pib_pc_mediano']:,.0f}, IDS={p['ids_medio']:.3f}")

Perfis regionais: 10 clusters
  No-0: 201 munic., PIB R$11,734, IDS=0.433
  No-1: 661 munic., PIB R$11,188, IDS=0.388
  No-2: 207 munic., PIB R$16,518, IDS=0.452
  No-3: 572 munic., PIB R$11,293, IDS=0.510
  No-4: 43 munic., PIB R$81,217, IDS=0.530
  Su-0: 494 munic., PIB R$37,765, IDS=0.413
  Su-1: 470 munic., PIB R$45,616, IDS=0.521
  Su-2: 177 munic., PIB R$44,255, IDS=0.475
  Ce-0: 140 munic., PIB R$33,357, IDS=0.480
  Ce-1: 284 munic., PIB R$39,027, IDS=0.463


---
## Comparação das 3 Linhas

In [11]:
# Silhueta global de cada abordagem
sil_base = silhouette_score(X_all, df['cluster_k4'])
sil_L1 = silhouette_score(X_all, df['subcluster_L1'])
sil_L2 = silhouette_score(X_all, df['cluster_k10'])
# L3: silhueta global não comparável diretamente (features diferentes por região)
# Usamos média ponderada das silhuetas regionais
sil_L3_pond = sum(
    resultados_L3[r]['silhouette'] * resultados_L3[r]['n_municipios'] 
    for r in regioes
) / len(df)

db_base = davies_bouldin_score(X_all, df['cluster_k4'])
db_L1 = davies_bouldin_score(X_all, df['subcluster_L1'])
db_L2 = davies_bouldin_score(X_all, df['cluster_k10'])

comparacao = {
    'Base K=4': {
        'n_clusters': 4,
        'silhouette': round(sil_base, 4),
        'davies_bouldin': round(db_base, 4),
        'max_cluster_pct': round(df['cluster_k4'].value_counts().max()/len(df)*100, 1),
        'min_cluster': int(df['cluster_k4'].value_counts().min()),
    },
    'Linha 1 - Hierárquica': {
        'n_clusters': n_subclusters_L1,
        'silhouette': round(sil_L1, 4),
        'davies_bouldin': round(db_L1, 4),
        'max_cluster_pct': round(df['subcluster_L1'].value_counts().max()/len(df)*100, 1),
        'min_cluster': int(df['subcluster_L1'].value_counts().min()),
    },
    'Linha 2 - K=10': {
        'n_clusters': 10,
        'silhouette': round(sil_L2, 4),
        'davies_bouldin': round(db_L2, 4),
        'max_cluster_pct': round(df['cluster_k10'].value_counts().max()/len(df)*100, 1),
        'min_cluster': int(df['cluster_k10'].value_counts().min()),
    },
    'Linha 3 - Regional': {
        'n_clusters': total_clusters_L3,
        'silhouette_ponderada': round(sil_L3_pond, 4),
        'max_cluster_pct': round(df['cluster_regional'].value_counts().max()/len(df)*100, 1),
        'min_cluster': int(df['cluster_regional'].value_counts().min()),
        'detalhes_por_regiao': {r: resultados_L3[r] for r in regioes}
    }
}

print('\n=== COMPARAÇÃO DAS 3 LINHAS ===')
print(f'{"Abordagem":<25} {"Clusters":>8} {"Silhueta":>10} {"DB":>8} {"Max%":>6} {"Min":>6}')
print('-' * 65)
for nome, m in comparacao.items():
    sil_val = m.get('silhouette', m.get('silhouette_ponderada', 0))
    db_val = m.get('davies_bouldin', '-')
    print(f'{nome:<25} {m["n_clusters"]:>8} {sil_val:>10.4f} {str(db_val):>8} {m["max_cluster_pct"]:>5.1f}% {m["min_cluster"]:>6}')

print(f'\nTotal de municípios em todas as abordagens: {len(df)} (0 perdidos)')


=== COMPARAÇÃO DAS 3 LINHAS ===
Abordagem                 Clusters   Silhueta       DB   Max%    Min
-----------------------------------------------------------------
Base K=4                         4     0.1585   1.6845  36.6%    597
Linha 1 - Hierárquica           16     0.0512   2.2876  16.3%     18
Linha 2 - K=10                  10     0.1215    1.587  13.9%    275
Linha 3 - Regional              14     0.2246        -  23.5%     43

Total de municípios em todas as abordagens: 5204 (0 perdidos)


In [12]:
# Salvar resultados
results = {
    'comparacao': comparacao,
    'perfis_L1': perfis_L1,
    'perfis_L2': {str(k): v for k, v in perfis_L2.items()},
    'perfis_L3': perfis_L3,
    'metricas_L1_por_grupo': metricas_L1,
    'resultados_L3_por_regiao': resultados_L3,
    'total_municipios': len(df),
    'total_outliers': int(df['is_outlier'].sum()),
    'nota': 'Outliers detectados via P1/P99, removidos para fit, reassociados ao cluster mais próximo via predict'
}

with open(os.path.join(OUT_RES, 'tres_linhas_pesquisa.json'), 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=str)

# Salvar CSV completo
df.to_csv(os.path.join(OUT_RES, 'municipios_3linhas.csv'), index=False)

print(f'Resultados salvos em:')
print(f'  {os.path.join(OUT_RES, "tres_linhas_pesquisa.json")}')
print(f'  {os.path.join(OUT_RES, "municipios_3linhas.csv")}')
print(f'\nColunas de cluster no CSV: cluster_k4, subcluster_L1, cluster_k10, cluster_regional, regiao_cluster_label')
print(f'Flag de outliers: is_outlier')

Resultados salvos em:
  c:\Users\guilhermecorrea\Downloads\Gui\Projetos\PortifolioProjetos\projeto_5\outputs\results\tres_linhas_pesquisa.json
  c:\Users\guilhermecorrea\Downloads\Gui\Projetos\PortifolioProjetos\projeto_5\outputs\results\municipios_3linhas.csv

Colunas de cluster no CSV: cluster_k4, subcluster_L1, cluster_k10, cluster_regional, regiao_cluster_label
Flag de outliers: is_outlier


In [13]:
# Verificação final
assert len(df) == 5204, f'Esperado 5204, obteve {len(df)}'
assert df['cluster_k4'].isna().sum() == 0, 'Há NAs em cluster_k4'
assert df['subcluster_L1'].min() >= 0, 'Subcluster L1 não atribuído'
assert df['cluster_k10'].isna().sum() == 0, 'Há NAs em cluster_k10'
assert df['cluster_regional'].min() >= 0, 'Cluster regional não atribuído'
print('✓ Todos os 5.204 municípios possuem cluster em todas as 3 linhas.')
print(f'✓ {df["is_outlier"].sum()} outliers sinalizados e reassociados.')

✓ Todos os 5.204 municípios possuem cluster em todas as 3 linhas.
✓ 298 outliers sinalizados e reassociados.
